In [ ]:
# autoreload
%load_ext autoreload
%autoreload 2

from scipy import signal
from scipy import interpolate
from scipy import ndimage
import numpy as np
import pycatch22 
from sktime.transformations.panel import catch22
import tsfresh
from tqdm import tqdm
import sys, os
import pandas as pd 
import dotenv
load_dotenv = dotenv.load_dotenv('../.env')
# load local library
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
import extractor
import preprocessing

import gc
import seaborn as sns
import matplotlib.pyplot as plt

from collections import defaultdict

from time import sleep

from scipy.stats import skew, kurtosis
from scipy.fft import rfft, rfftfreq
from scipy.signal import cwt, ricker
import pymannkendall as mk

# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AffinityPropagation, SpectralClustering, HDBSCAN
from sklearn.mixture import GaussianMixture as GMM
from umap import UMAP
from scaler import GaussRankScaler
from sklearn.pipeline import Pipeline

import nolds


In [ ]:
AKI_PATH = os.environ['AKI_PATH']
os.chdir(AKI_PATH)

In [ ]:
ts_data = pd.read_parquet("ts.parquet.snappy")

In [ ]:
ts_data = ts_data.assign(ID=ts_data.ID.astype('int64'))

## Exploration

In [ ]:
min_days_length = 1*365
maxts = ts_data.groupby('ID').Time_days.max()
ltids = maxts[maxts>min_days_length].index

print(f'There are {ltids.shape[0]} patients (or {round(100*ltids.shape[0]/maxts.shape[0], 2)}%) with minimally {min_days_length} of measurements')

## Preprocess

In [ ]:
RangeList = [90, 180] + list(np.arange(1*365,10*365, 365))
WINDOW_SIZE = 3 # for smoothing
MIN_MEASUREMENTS = 5
SMOOTHING_TYPE = 'gaussian_kernel' # rolling_mean or gaussian_kernel
STANDARDIZE_TS = True
TIME_RESOLUTION = 14 # days, for interpolation.

RangeDFDict = {}
RangeDFDict_smoothed = {}
for MinDays in RangeList:
    print(f'Filtering for min days: {MinDays}')
    ts_temp = preprocessing.get_filtered_df(ts_data.copy(), 
                                            id_col='ID', 
                                            time_col='Time_days',
                                            min_days=MinDays, 
                                            min_measurements=MIN_MEASUREMENTS)
    
    if STANDARDIZE_TS:
        print(f'Standardizing for min days: {MinDays}')
        ts_temp = preprocessing.normalise_ts(ts_temp, 
                                             id_col='ID', 
                                             time_col='Time_days',
                                             val_col='eGFR_CKDEpi2012',
                                             df_out=True)
    
    
    print(f'Interpolating for min days: {MinDays}')
    RangeDFDict[MinDays] = preprocessing.get_interpolated(ts_temp, id_col='ID', time_col='Time_days',
                                              max_days=MinDays,
                                              val_col='eGFR_CKDEpi2012',
                                              time_res=TIME_RESOLUTION,
                                              df_out=True)
    
    print(f"Smoothing for min days: {MinDays} with window {WINDOW_SIZE}")
    if SMOOTHING_TYPE == 'rolling_mean':
        RangeDFDict_smoothed[MinDays] = preprocessing.get_smoothed_rolling_mean(RangeDFDict[MinDays],
                                                        id_col='ID', time_col='Time_days',
                                                        val_col='eGFR_CKDEpi2012', 
                                                        window=WINDOW_SIZE, 
                                                        df_out=True)
    elif SMOOTHING_TYPE == 'gaussian_kernel':
        RangeDFDict_smoothed[MinDays] = preprocessing.get_smoothed_gaussian_kernel(RangeDFDict[MinDays],
                                                        id_col='ID', time_col='Time_days',
                                                        val_col='eGFR_CKDEpi2012', 
                                                        window=WINDOW_SIZE,
                                                        df_out=True)
    

In [ ]:
CrossSectDict = {}
# get_smoothNsmooth_diffStatistics
for MinDays in RangeList:
    print(f'Getting cross-sectional features for min days, for the smoothed set, with periods of: {MinDays} days')
    tsS = RangeDFDict_smoothed[MinDays]
    smoothed_cross = extractor.get_crossectional(tsS, 
                                        id_col='ID',
                                        val_col='eGFR_CKDEpi2012',
                                        time_col='Time_days',
                                        tsfresh_features=False,
                                        catch22_features=True,
                                        cesium_features=False,
                                        antropy_features=False,
                                        nolds_features=False)
    ############
    print(f'Getting cross-sectional features for min days, for the raw set, with periods of: {MinDays} days')
    tsR = RangeDFDict[MinDays]
    raw_cross = extractor.get_crossectional(tsR, 
                                        id_col='ID',
                                        val_col='eGFR_CKDEpi2012',
                                        time_col='Time_days',
                                        tsfresh_features=False,
                                        catch22_features=True,
                                        cesium_features=False,
                                        antropy_features=False,
                                        nolds_features=False)
    ############
    print(f'Merging cross-sectional features for min days, the raw set, with periods of: {MinDays} days')
    merged_cross = smoothed_cross.merge(raw_cross,
                                        left_index=True, 
                                        right_index=True, 
                                        suffixes=('_smoothed', '_raw'))
    smoothNsmoothStats = extractor.get_smoothNsmooth_diffStatistics(tsS,tsR, 
                                                                    id_col='ID',
                                                                    val_col='eGFR_CKDEpi2012',
                                                                    time_col='Time_days')
    
    smoothed_cross = smoothed_cross.merge(smoothNsmoothStats, left_index=True, right_index=True)
    raw_cross = raw_cross.merge(smoothNsmoothStats, left_index=True, right_index=True)
    merged_cross = merged_cross.merge(smoothNsmoothStats, left_index=True, right_index=True)

    CrossSectDict[MinDays] = {
                              'smoothed': smoothed_cross,
                              'raw': raw_cross,
                              'merged': merged_cross
                            }
    
    gc.collect()

## Remove redundant features

In [ ]:
# remove features that have more than N% of missing values
MAX_MISSING_VALUES_PER_COLUMN = 0.5

print("Removing features for all periods that are too sparse")
num_rem = []
for period in tqdm(CrossSectDict.keys()):
    for key in CrossSectDict[period].keys():
        FINAL_FEATURES = CrossSectDict[period][key]
        missing_ratio = FINAL_FEATURES.isna().sum(axis=0)/\
                            FINAL_FEATURES.shape[0]
        missing_cols = missing_ratio[missing_ratio>MAX_MISSING_VALUES_PER_COLUMN].index
        
        CrossSectDict[period][key] = FINAL_FEATURES.drop(columns=missing_cols)
        num_rem.append(missing_cols.shape[0])
        
print(f"Number of columns removed: {num_rem}")

In [ ]:
# remove plus or minus infinity values
print("Removing infinite values")
num_rem = []
for period in tqdm(CrossSectDict.keys()):
    for key in CrossSectDict[period].keys():
        FINAL_FEATURES = CrossSectDict[period][key]
        inf_cols = FINAL_FEATURES.columns[FINAL_FEATURES.isin([np.inf, -np.inf]).any()]
        CrossSectDict[period][key] = FINAL_FEATURES.drop(columns=inf_cols)
        num_rem.append(inf_cols.shape[0])

In [ ]:
from sklearn.feature_selection import VarianceThreshold

print("Removing zero variance features for all periods...")
num_zero = []
for period in tqdm(CrossSectDict.keys()):
    for key in CrossSectDict[period].keys():
        FINAL_FEATURES = CrossSectDict[period][key]
        
        var_thresh = VarianceThreshold(threshold=0.)
        var_thresh.fit(FINAL_FEATURES)
        variances = var_thresh.variances_

        # remove features with zero variance
        zero_variance_features = FINAL_FEATURES.columns[variances == 0]
        non_zero_variance_features = FINAL_FEATURES.columns[variances > 0]
        FINAL_FEATURES = FINAL_FEATURES.drop(zero_variance_features, axis=1)
        
        CrossSectDict[period][key] = FINAL_FEATURES
        num_zero.append(zero_variance_features.shape[0])  
        
print(f"Number of columns removed: {num_zero}")

In [ ]:
# remove features that are highly correlated
print("Removing perfectly correlated features for all periods...")
num_corr = []
duplicated = set()
for period in tqdm(CrossSectDict.keys()):
    for key in CrossSectDict[period].keys():
        FINAL_FEATURES = CrossSectDict[period][key]
        
        #dist_matrix = 1-FINAL_FEATURES.corr(method='spearman').abs()
        #cols = dist_matrix.columns        
        cols = FINAL_FEATURES.columns
        droplist = []
        for i,cl in enumerate(cols):
            for cr in cols[i+1:]:
                if FINAL_FEATURES[cl].equals(FINAL_FEATURES[cr]):
                    droplist.append(cr)
                    duplicated.add((cl, cr))
        to_drop = list(set(droplist))
                
        FINAL_FEATURES = FINAL_FEATURES.drop(columns=to_drop)
        CrossSectDict[period][key] = FINAL_FEATURES
        num_corr.append(len(to_drop))
print(f"Number of columns removed: {num_corr}")

In [ ]:
print(f"Duplicate column-pairs: {duplicated}")


In [ ]:
CrossSectDict[period][key] 

## Imputation

In [ ]:
#imputer = SimpleImputer(strategy='mean')
imputer = KNNImputer(n_neighbors=7, weights='distance')

## Scale

In [ ]:
#scaler = GaussRankScaler()
#scaler = QuantileTransformer(output_distribution='normal')
scaler = StandardScaler()
reducer = PCA(n_components=100)
#reducer = UMAP(n_components=6, n_neighbors=15, min_dist=0., metric='manhattan')

## Cluster

In [ ]:
#clusterer = HDBSCAN(min_cluster_size=20, min_samples=15, cluster_selection_epsilon=0.5)
#clusterer = KMeans(n_clusters=3)
clusterer = GMM(n_clusters=3)

# Run pipeline

In [ ]:
ModelDict = {}
for period in tqdm(CrossSectDict.keys()):
    ModelDict[period] = {}
    for preptype in CrossSectDict[period].keys():
        le_pipe_clusterer = Pipeline([
                                      ('scaler', scaler),
                                      ('imputer', imputer),
                                      ('reducer', reducer), 
                                      ('clusterer', clusterer)],
                                     verbose=True)

        le_pipe_clusterer.fit(CrossSectDict[period][preptype])
        CrossSectDict[period][preptype]['cluster'] = le_pipe_clusterer.named_steps['clusterer'].labels_
        CrossSectDict[period][preptype]['cluster_proba'] = le_pipe_clusterer.named_steps['clusterer'].probabilities_
        ModelDict[period][preptype] = le_pipe_clusterer

## Plot

In [ ]:
for period in tqdm(CrossSectDict.keys()):
    for preptype in CrossSectDict[period].keys():
        RangeDFDict_smoothed[period] = \
            pd.merge(RangeDFDict_smoothed[period],
                    CrossSectDict[period][preptype][['cluster']], 
                    left_on='ID', 
                    right_index=True, 
                    how='left',
                    suffixes = ('', f'_{preptype}')
                )

In [ ]:
for period in tqdm(CrossSectDict.keys()):
    for preptype in CrossSectDict[period].keys():
        #RangeDFDict[period] = pd.DataFrame(RangeDFDict[period])
        RangeDFDict[period] = \
            pd.merge(RangeDFDict[period],
                    CrossSectDict[period][preptype][['cluster']], 
                    left_on='ID', 
                    right_index=True, 
                    how='left',
                    suffixes = ('', f'_{preptype}')
                )

In [ ]:
# cluster_smoothed, cluster_raw, cluster_merged
RangeDFDict_smoothed.keys()

In [ ]:
sns.lineplot(data=RangeDFDict_smoothed[365], 
             x='Time_days',
             y='eGFR_CKDEpi2012',
             hue='cluster_merged',
             err_style='band',
             errorbar='ci',
             )

In [ ]:
sns.relplot(data=RangeDFDict_smoothed[365], 
             x='Time_days',
             y='eGFR_CKDEpi2012',
             hue='ID',
             kind='line',
             col='cluster_merged',
             col_wrap=2
             )